## Reaction window accuracy 

In [0]:

# GOLD KPI 1 — Reaction Window Accuracy

# nifty_t1_return (T+1), nifty_t3_return (T+3), nifty_t5_return (T+5)
# computed as compound returns over exactly 1, 3, 5 trading days
# post-event using: (close_T+N / close_T - 1) * 100


from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import DoubleType

spark.sql("USE CATALOG iran_israel_capstone_project")

# Read silver layer
silver = spark.table("silver.daily_market_clean")

# Window ordered by trade_date for LEAD calculations
date_win = Window.orderBy("trade_date")

# Get future close prices at exactly T+1, T+3, T+5 trading days
silver_with_leads = (
    silver
    .withColumn("nifty_close_t1", F.lead("nifty_close", 1).over(date_win))
    .withColumn("nifty_close_t3", F.lead("nifty_close", 3).over(date_win))
    .withColumn("nifty_close_t5", F.lead("nifty_close", 5).over(date_win))
)

# Compute compound returns (percentage) for event rows only
event_reactions = (
    silver_with_leads
    .filter(F.col("event_id").isNotNull())
    .withColumn(
        "nifty_t1_return",
        F.round((F.col("nifty_close_t1") / F.col("nifty_close") - 1) * 100, 4)
    )
    .withColumn(
        "nifty_t3_return",
        F.round((F.col("nifty_close_t3") / F.col("nifty_close") - 1) * 100, 4)
    )
    .withColumn(
        "nifty_t5_return",
        F.round((F.col("nifty_close_t5") / F.col("nifty_close") - 1) * 100, 4)
    )
    .select(
        "trade_date", "event_id", "event_type", "severity",
        "t_plus_1_expected", "nifty_close",
        "nifty_close_t1", "nifty_close_t3", "nifty_close_t5",
        "nifty_t1_return", "nifty_t3_return", "nifty_t5_return",
    )
    .orderBy("trade_date")
)

print(f"Event rows with reaction windows: {event_reactions.count()}")
display(event_reactions)

In [0]:

# SPOT-CHECK — Verify 3 events manually

# For each event, show T+0 through T+5 close prices from the
# raw silver data, then compare against the computed returns.

spot_events = ["IRN-ISR-002", "HORMUZ-002", "USA-IRN-003"]

for eid in spot_events:
    # Get event date
    evt_row = event_reactions.filter(F.col("event_id") == eid).first()
    if evt_row is None:
        print(f"\n Event {eid} not found in event_reactions")
        continue

    evt_date = evt_row["trade_date"]
    print(f"\n{'='*60}")
    print(f"Event: {eid}  |  Date: {evt_date}  |  Type: {evt_row['event_type']}")
    print(f"{'='*60}")

    # Get T+0 through T+5 trading days from silver
    future_days = (
        silver
        .filter(F.col("trade_date") >= evt_date)
        .orderBy("trade_date")
        .select("trade_date", "nifty_close")
        .limit(6)  # T+0 through T+5
        .collect()
    )

    close_t0 = future_days[0]["nifty_close"]
    print(f"\n  Raw close prices (trading days):")
    for i, row in enumerate(future_days):
        label = f"T+{i}"
        ret = (row['nifty_close'] / close_t0 - 1) * 100 if close_t0 else None
        print(f"    {label}: {row['trade_date']}  close={row['nifty_close']:.2f}  cum_return={ret:+.4f}%")

    # Compare with computed values
    print(f"\n  Computed returns from gold:")
    print(f"    nifty_t1_return: {evt_row['nifty_t1_return']}%")
    print(f"    nifty_t3_return: {evt_row['nifty_t3_return']}%")
    print(f"    nifty_t5_return: {evt_row['nifty_t5_return']}%")

    # Validate T+1
    if len(future_days) >= 2:
        expected_t1 = round((future_days[1]['nifty_close'] / close_t0 - 1) * 100, 4)
        match_t1 = "" if abs(expected_t1 - (evt_row['nifty_t1_return'] or 0)) < 0.001 else "x"
        print(f"    T+1 manual check: {expected_t1}%  {match_t1}")
    if len(future_days) >= 4:
        expected_t3 = round((future_days[3]['nifty_close'] / close_t0 - 1) * 100, 4)
        match_t3 = "" if abs(expected_t3 - (evt_row['nifty_t3_return'] or 0)) < 0.001 else "x"
        print(f"    T+3 manual check: {expected_t3}%  {match_t3}")
    if len(future_days) >= 6:
        expected_t5 = round((future_days[5]['nifty_close'] / close_t0 - 1) * 100, 4)
        match_t5 = "" if abs(expected_t5 - (evt_row['nifty_t5_return'] or 0)) < 0.001 else "x"
        print(f"    T+5 manual check: {expected_t5}%  {match_t5}")

print("\n" + "="*60)
print(" Spot-check complete")

In [0]:

# Save event_reactions to gold schema

spark.sql("CREATE SCHEMA IF NOT EXISTS iran_israel_capstone_project.gold")

(
    event_reactions.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("iran_israel_capstone_project.gold.event_reaction_windows")
)

print(f" gold.event_reaction_windows written ({event_reactions.count()} rows)")

In [0]:
# DRONE_ATTACK showing increasing T+1 → T+3 → T+5 returns is actually correct and real
# Your chart shows DRONE_ATTACK at approximately T+1: ~0.39%, T+3: ~0.39%, T+5: +1.3% — an improving trend. This is historically accurate for a specific reason:
# Drone attacks are perceived as "controlled" escalations. Markets initially panic at T+1, but within 3–5 trading days they realise:

# Drone attacks in the Iran–Israel context were largely intercepted (Israel claimed 99% interception of the April 13, 2024 attack)
# They signal Iran's reluctance to escalate to full missile/ground war
# The "worst case" (Strait of Hormuz closure, full war) didn't materialise

# This is confirmed by real data: on April 19, 2024 — just 6 days after Iran's drone attack — Nifty closed at 22,147, sharply higher, after Iran said it wouldn't consider "immediate" retaliation against Israel, with short-covering by foreign investors driving a recovery.

## Direction prediction accuracy

In [0]:
# there's no code-level fix that would be appropriate — recalculating t_plus_1_expected would defeat the KPI's purpose of testing "analytical pre-work quality." The 41.7% accuracy is the genuine result: the team's pre-analysis was significantly biased toward MARKET_UP when geopolitical events actually triggered short-term selloffs.

In [0]:

# GOLD KPI 2 — Direction Prediction Accuracy

# Compare t_plus_1_expected (team pre-analysis) vs actual
# nifty_t1_return sign.  Target >= 60%.


from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark.sql("USE CATALOG iran_israel_capstone_project")
event_reactions = spark.table("gold.event_reaction_windows")

# ── Classify each prediction as correct / incorrect ──────────
direction_df = (
    event_reactions
    .filter(F.col("nifty_t1_return").isNotNull())
    .withColumn(
        "actual_direction",
        F.when(F.col("nifty_t1_return") > 0, F.lit("MARKET_UP"))
         .when(F.col("nifty_t1_return") < 0, F.lit("MARKET_DOWN"))
         .otherwise(F.lit("FLAT"))
    )
    .withColumn(
        "prediction_correct",
        F.when(
            F.col("t_plus_1_expected") == F.col("actual_direction"), 1
        ).otherwise(0)
    )
    .select(
        "trade_date", "event_id", "event_type", "severity",
        "t_plus_1_expected", "nifty_t1_return",
        "actual_direction", "prediction_correct",
    )
    .orderBy("trade_date")
)

# ── Compute overall accuracy ─────────────────────────────────
total_events = direction_df.count()
correct_count = direction_df.filter(F.col("prediction_correct") == 1).count()
accuracy_pct = correct_count / total_events * 100

status = "" if accuracy_pct >= 60 else "x"
print(f"\n{'='*60}")
print(f"  DIRECTION PREDICTION ACCURACY")
print(f"{'='*60}")
print(f"  Total events evaluated : {total_events}")
print(f"  Correct predictions    : {correct_count}")
print(f"  Accuracy               : {accuracy_pct:.1f}%")
print(f"  Target                 : >= 60%")
print(f"  Status                 : {status}")
print(f"{'='*60}\n")

# ── Accuracy by event type ───────────────────────────────────
print("── Accuracy by Event Type ──")
(
    direction_df
    .groupBy("event_type")
    .agg(
        F.count("*").alias("events"),
        F.sum("prediction_correct").alias("correct"),
        F.round(F.avg("prediction_correct") * 100, 1).alias("accuracy_pct"),
    )
    .orderBy(F.desc("accuracy_pct"))
    .show(truncate=False)
)

# ── Accuracy by severity ─────────────────────────────────────
print("── Accuracy by Severity ──")
(
    direction_df
    .groupBy("severity")
    .agg(
        F.count("*").alias("events"),
        F.sum("prediction_correct").alias("correct"),
        F.round(F.avg("prediction_correct") * 100, 1).alias("accuracy_pct"),
    )
    .orderBy(F.desc("accuracy_pct"))
    .show(truncate=False)
)

# ── Per-event detail ─────────────────────────────────────────
display(direction_df)

# ── Persist to gold ──────────────────────────────────────────
(
    direction_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("iran_israel_capstone_project.gold.direction_prediction_accuracy")
)
print(f" gold.direction_prediction_accuracy written ({total_events} rows)")

## Severity Correlation

In [0]:

# GOLD KPI 3 — Severity Correlation

# HIGH/CRITICAL events must show a more negative median
# nifty_t1_return than LOW/MEDIUM events.


from pyspark.sql import functions as F

spark.sql("USE CATALOG iran_israel_capstone_project")
event_reactions = spark.table("gold.event_reaction_windows")

# ── GROUP BY severity: median + mean T+1 return 
severity_agg = (
    event_reactions
    .filter(F.col("nifty_t1_return").isNotNull())
    .groupBy("severity")
    .agg(
        F.count("*").alias("events"),
        F.round(F.percentile_approx("nifty_t1_return", 0.5), 4).alias("median_t1_return"),
        F.round(F.avg("nifty_t1_return"), 4).alias("mean_t1_return"),
        F.round(F.min("nifty_t1_return"), 4).alias("min_t1_return"),
        F.round(F.max("nifty_t1_return"), 4).alias("max_t1_return"),
    )
)

print("── Nifty T+1 Return by Severity Level ──")
severity_agg.orderBy("severity").show(truncate=False)

# ── Bucket into HIGH_CRITICAL vs LOW_MEDIUM 
bucketed = (
    event_reactions
    .filter(F.col("nifty_t1_return").isNotNull())
    .withColumn(
        "severity_bucket",
        F.when(F.col("severity").isin("HIGH", "CRITICAL"), "HIGH_CRITICAL")
         .otherwise("LOW_MEDIUM")
    )
)

bucket_agg = (
    bucketed
    .groupBy("severity_bucket")
    .agg(
        F.count("*").alias("events"),
        F.round(F.percentile_approx("nifty_t1_return", 0.5), 4).alias("median_t1_return"),
        F.round(F.avg("nifty_t1_return"), 4).alias("mean_t1_return"),
    )
    .orderBy("severity_bucket")
)

bucket_rows = bucket_agg.collect()
bucket_agg.show(truncate=False)

# ── Validate KPI rule
high_crit = [r for r in bucket_rows if r["severity_bucket"] == "HIGH_CRITICAL"][0]
low_med   = [r for r in bucket_rows if r["severity_bucket"] == "LOW_MEDIUM"][0]

rule_pass = high_crit["median_t1_return"] < low_med["median_t1_return"]
status = "" if rule_pass else "x"

print(f"\n{'='*60}")
print(f"  SEVERITY CORRELATION CHECK")
print(f"{'='*60}")
print(f"  HIGH/CRITICAL median T+1 : {high_crit['median_t1_return']:+.4f}%  ({high_crit['events']} events)")
print(f"  LOW/MEDIUM    median T+1 : {low_med['median_t1_return']:+.4f}%  ({low_med['events']} events)")
print(f"  Difference               : {high_crit['median_t1_return'] - low_med['median_t1_return']:+.4f} pp")
print(f"  Rule (HC < LM)           : {status}")
print(f"{'='*60}")

# ── Persist to gold ──────────────────────────────────────────
severity_detail = (
    bucketed
    .select(
        "trade_date", "event_id", "event_type", "severity",
        "severity_bucket", "nifty_t1_return",
        "nifty_t3_return", "nifty_t5_return",
    )
    .orderBy("trade_date")
)

(
    severity_detail.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("iran_israel_capstone_project.gold.severity_correlation")
)
print(f"\n gold.severity_correlation written ({severity_detail.count()} rows)")

display(severity_detail)

In [0]:

# Build gold.gold_event_market_reaction

# Comprehensive event-level table with multi-asset reaction
# windows and direction prediction accuracy.


from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark.sql("USE CATALOG iran_israel_capstone_project")
silver = spark.table("silver.daily_market_clean")

date_win = Window.orderBy("trade_date")

# LEAD for all assets: nifty, brent, usdinr, vix + future dates
with_leads = (
    silver
    # Nifty leads
    .withColumn("nifty_close_t1", F.lead("nifty_close", 1).over(date_win))
    .withColumn("nifty_close_t3", F.lead("nifty_close", 3).over(date_win))
    .withColumn("nifty_close_t5", F.lead("nifty_close", 5).over(date_win))
    # Brent lead
    .withColumn("brent_close_t1", F.lead("brent_close", 1).over(date_win))
    # USDINR lead
    .withColumn("usdinr_close_t1", F.lead("usdinr_close", 1).over(date_win))
    # India VIX lead
    .withColumn("vix_close_t1", F.lead("indiavix_close", 1).over(date_win))
    # Future dates
    .withColumn("t1_date", F.lead("trade_date", 1).over(date_win))
    .withColumn("t3_date", F.lead("trade_date", 3).over(date_win))
    .withColumn("t5_date", F.lead("trade_date", 5).over(date_win))
)

# Filter to event rows and compute returns
gold_reaction = (
    with_leads
    .filter(F.col("event_id").isNotNull())
    # Nifty returns
    .withColumn("nifty_t1_return",
        F.round((F.col("nifty_close_t1") / F.col("nifty_close") - 1) * 100, 4))
    .withColumn("nifty_t3_return",
        F.round((F.col("nifty_close_t3") / F.col("nifty_close") - 1) * 100, 4))
    .withColumn("nifty_t5_return",
        F.round((F.col("nifty_close_t5") / F.col("nifty_close") - 1) * 100, 4))
    # Brent T+1 return
    .withColumn("brent_t1_return",
        F.round((F.col("brent_close_t1") / F.col("brent_close") - 1) * 100, 4))
    # USDINR T+1 change
    .withColumn("usdinr_t1_change",
        F.round((F.col("usdinr_close_t1") / F.col("usdinr_close") - 1) * 100, 4))
    # VIX T+1 change
    .withColumn("vix_t1_change",
        F.round((F.col("vix_close_t1") / F.col("indiavix_close") - 1) * 100, 4))
    # Direction prediction
    .withColumn("actual_t1_direction",
        F.when(F.col("nifty_t1_return") > 0, "MARKET_UP")
         .when(F.col("nifty_t1_return") < 0, "MARKET_DOWN")
         .otherwise("FLAT"))
    .withColumn("direction_prediction_correct",
        F.col("t_plus_1_expected") == F.col("actual_t1_direction"))
    # Select final columns
    .select(
        "trade_date", "event_id", "event_type", "severity", "crude_risk",
        "t_plus_1_expected",
        F.col("nifty_close").alias("nifty_event_close"),
        F.col("nifty_daily_return_pct").alias("nifty_event_day_return"),
        "nifty_t1_return", "nifty_t3_return", "nifty_t5_return",
        "t1_date", "t3_date", "t5_date",
        F.col("brent_close").alias("brent_event_close"),
        F.col("brent_daily_change_pct").alias("brent_event_day_change"),
        "brent_t1_return",
        F.col("usdinr_close").alias("usdinr_event_close"),
        "usdinr_t1_change",
        F.col("indiavix_close").alias("vix_event_close"),
        "vix_t1_change",
        "actual_t1_direction", "direction_prediction_correct",
    )
    .orderBy("trade_date")
)

# Persist
(
    gold_reaction.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("iran_israel_capstone_project.gold.gold_event_market_reaction")
)

row_count = gold_reaction.count()
print(f"gold.gold_event_market_reaction written ({row_count} rows)")
display(gold_reaction)